# Curriculum scrape sources

This notebook loads the same module list the enrichment pipeline uses: `MODULES` plus `PIPELINE_ONLY_MODULES` and any `load_dynamic_pipeline_only_modules()` entries from [`build_study_data.py`](../build_study_data.py).

It only **loads and displays** resource metadata (URLs and local export paths)—no scraping or LLM calls.

Run cells from the repository root (`study-app/`) or from `docs/`; the first code cell discovers the repo by locating `build_study_data.py` and `pipeline/`.

In [8]:
from __future__ import annotations

import importlib.util
from pathlib import Path

try:
    from IPython.display import display
except ImportError:  # pragma: no cover
    display = print


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "build_study_data.py").exists() and (candidate / "pipeline").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from inside the study-app repository "
        "(expected build_study_data.py and pipeline/ directory)."
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
BUILD_STUDY_DATA = REPO_ROOT / "build_study_data.py"


def load_pipeline_modules() -> list[dict]:
    """Match pipeline/runner._load_modules (without relying on config REPO_ROOT)."""
    spec = importlib.util.spec_from_file_location("build_study_data", BUILD_STUDY_DATA)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot load {BUILD_STUDY_DATA}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    public = list(module.MODULES)
    pipeline_only = list(getattr(module, "PIPELINE_ONLY_MODULES", []))
    dynamic_loader = getattr(module, "load_dynamic_pipeline_only_modules", None)
    if callable(dynamic_loader):
        pipeline_only.extend(dynamic_loader())
    return public + pipeline_only


modules = load_pipeline_modules()
resource_rows = sum(len(m.get("resources") or []) for m in modules)

print(f"Repo root: {REPO_ROOT}")
print(f"build_study_data.py: {BUILD_STUDY_DATA}")
print(f"Modules: {len(modules)} | Resource rows: {resource_rows}")

Repo root: /Users/chetangoel/Desktop/Repositories/study-app
build_study_data.py: /Users/chetangoel/Desktop/Repositories/study-app/build_study_data.py
Modules: 43 | Resource rows: 390


In [9]:
from urllib.parse import urlparse

rows: list[dict] = []
for m in modules:
    for r in m.get("resources") or []:
        url = str(r.get("url") or "")
        sp = r.get("sourcePath")
        rows.append(
            {
                "module_id": m.get("id"),
                "module_title": m.get("title"),
                "phase": m.get("phase"),
                "track": m.get("track"),
                "label": r.get("label"),
                "url": url,
                "source_path": str(sp) if sp else "",
                "scrape_kind": "local_export" if sp else "remote_url",
            }
        )

try:
    import pandas as pd

    df = pd.DataFrame(rows)
    df = df.sort_values(["module_id", "label"], kind="stable").reset_index(drop=True)
    pd.set_option("display.max_rows", 200)
    pd.set_option("display.max_colwidth", 80)
    display(df)
except ImportError:
    import json

    print("Install pandas for a sortable table (pip install pandas). Preview:\n")
    print(json.dumps(rows[:20], indent=2, ensure_ascii=False))
    print(f"\n... {len(rows)} total rows")

,module_id,module_title,phase,track,label,url,source_path,scrape_kind
0,arrays-linked-lists,Linear structures: arrays and linked lists,Core Track,dsa-leetcode,Arrays CS50,https://www.youtube.com/watch?v=tI_tIZFyKBw&t=3009s,,remote_url
1,arrays-linked-lists,Linear structures: arrays and linked lists,Core Track,dsa-leetcode,Dynamic Arrays,https://www.coursera.org/lecture/data-structures/dynamic-arrays-EwbnV,,remote_url
2,arrays-linked-lists,Linear structures: arrays and linked lists,Core Track,dsa-leetcode,Linked Lists CS50,https://www.youtube.com/watch?v=2T-A_GFuoTo&t=650s,,remote_url
3,arrays-linked-lists,Linear structures: arrays and linked lists,Core Track,dsa-leetcode,Linked Lists vs Arrays,https://www.coursera.org/lecture/data-structures-optimizing-performance/core...,,remote_url
4,arrays-linked-lists,Linear structures: arrays and linked lists,Core Track,dsa-leetcode,Pointers to Pointers,https://www.eskimo.com/~scs/cclass/int/sx8.html,,remote_url
...,...,...,...,...,...,...,...,...
385,trees-heaps,"Trees, BSTs, and heaps",Core Track,dsa-leetcode,Binary Search Tree Review,https://www.youtube.com/watch?v=x6At0nzX92o&index=1&list=PLA5Lqm4uh9Bbq-E0Zn...,,remote_url
386,trees-heaps,"Trees, BSTs, and heaps",Core Track,dsa-leetcode,Heap Review Playlist,https://www.youtube.com/playlist?list=PL9xmBV_5YoZNsyqgPW-DNwUeT8F8uhWc6,,remote_url
387,trees-heaps,"Trees, BSTs, and heaps",Core Track,dsa-leetcode,Intro to Trees,https://www.coursera.org/lecture/data-structures/trees-95qda,,remote_url
388,trees-heaps,"Trees, BSTs, and heaps",Core Track,dsa-leetcode,MIT Binary Heaps,https://www.youtube.com/watch?v=Xnpo1atN-Iw&list=PLUl4u3cNGP63EdVPNLG3ToM6La...,,remote_url


In [10]:
from collections import Counter

urls = [r["url"] for r in rows if r["url"]]
by_url = Counter(urls)
dup_urls = sorted((u, n) for u, n in by_url.items() if n > 1)

print(f"Rows with a URL: {len(urls)}")
print(f"Unique URLs: {len(by_url)} (pipeline dedupes by URL; first module wins)")
if dup_urls:
    print(f"\nDuplicate URL count: {len(dup_urls)}")
    for u, n in dup_urls[:25]:
        print(f"  {n}x {u}")
    if len(dup_urls) > 25:
        print(f"  ... and {len(dup_urls) - 25} more")

local = sum(1 for r in rows if r["scrape_kind"] == "local_export")
remote = len(rows) - local
print(f"\nLocal export rows (sourcePath): {local}")
print(f"Remote URL rows: {remote}")

hosts = Counter(urlparse(u).netloc for u in urls if u)
print("\nTop hosts (by row count):")
for host, n in hosts.most_common(15):
    print(f"  {n:4}  {host or '(empty)'}")

Rows with a URL: 390
Unique URLs: 260 (pipeline dedupes by URL; first module wins)

Duplicate URL count: 130
  2x https://leetcode.com/explore/interview/card/leetcodes-interview-crash-course-data-structures-and-algorithms/703/arraystrings/4500/
  2x https://leetcode.com/explore/interview/card/leetcodes-interview-crash-course-data-structures-and-algorithms/703/arraystrings/4501/
  2x https://leetcode.com/explore/interview/card/leetcodes-interview-crash-course-data-structures-and-algorithms/703/arraystrings/4502/
  2x https://leetcode.com/explore/interview/card/leetcodes-interview-crash-course-data-structures-and-algorithms/703/arraystrings/4503/
  2x https://leetcode.com/explore/interview/card/leetcodes-interview-crash-course-data-structures-and-algorithms/703/arraystrings/4504/
  2x https://leetcode.com/explore/interview/card/leetcodes-interview-crash-course-data-structures-and-algorithms/703/arraystrings/4505/
  2x https://leetcode.com/explore/interview/card/leetcodes-interview-crash-

In [11]:
import sys

sys.path.insert(0, str(REPO_ROOT / "pipeline"))
from config import pipeline_url_excluded_from_stage1_seeds
from url_classifier import UrlType, classify


def _is_youtube_host(url: str) -> bool:
    host = urlparse(url).netloc.casefold().removeprefix("www.")
    return host in {"youtube.com", "youtu.be"}


def _keep_url_for_stage1(url: str) -> bool:
    """Match docs/content-pipeline-debug Stage 1 seed filter (GitHub + LeetCode, no YouTube)."""
    if pipeline_url_excluded_from_stage1_seeds(url):
        return False
    if _is_youtube_host(url):
        return False
    t = classify(url)
    if t in {UrlType.YOUTUBE_VIDEO, UrlType.YOUTUBE_PLAYLIST}:
        return False
    if t in {UrlType.GITHUB_MARKDOWN, UrlType.GITHUB_PDF, UrlType.GITHUB_REPO}:
        return True
    if t == UrlType.LEETCODE:
        return True
    return False


# First curriculum occurrence wins (same as pipeline URL dedupe).
parent_rows: list[dict] = []
seen_urls: set[str] = set()
for r in rows:
    url = (r.get("url") or "").strip()
    if not url or r.get("scrape_kind") != "remote_url":
        continue
    if url in seen_urls:
        continue
    if not _keep_url_for_stage1(url):
        continue
    seen_urls.add(url)
    ut = classify(url)
    parent_rows.append(
        {
            "url": url,
            "url_type": ut.value,
            "module_id": r.get("module_id"),
            "module_title": r.get("module_title"),
            "module_phase": r.get("phase"),
            "label": r.get("label"),
        }
    )

print(f"Stage 1-style parent seeds (GitHub + LeetCode, no YouTube): {len(parent_rows)}")
print("(Deduped by URL; first module in curriculum order wins.)")
print(
    "Skipping URLs matching config.PIPELINE_STAGE1_EXCLUDED_URL_SUBSTRINGS "
    "(LeetCode DSA crash-course card — already scraped; still in app curriculum).\n"
)

for i, p in enumerate(parent_rows, start=1):
    print(f"{i:3}. [{p['url_type']}] {p['module_id']} — {p['label']}")
    print(f"     {p['url']}\n")

try:
    import pandas as pd

    parents_df = pd.DataFrame(parent_rows)
    parents_df = parents_df.sort_values(["url_type", "module_id", "label"], kind="stable").reset_index(
        drop=True
    )
    pd.set_option("display.max_rows", 300)
    pd.set_option("display.max_colwidth", 100)
    display(parents_df)
except ImportError:
    pass


Stage 1-style parent seeds (GitHub + LeetCode, no YouTube): 18
(Deduped by URL; first module in curriculum order wins.)
Skipping URLs matching config.PIPELINE_STAGE1_EXCLUDED_URL_SUBSTRINGS (LeetCode DSA crash-course card — already scraped; still in app curriculum).

  1. [github_markdown] setup-habits — Choose a Programming Language
     https://github.com/jwasham/coding-interview-university/blob/master/README.md#choose-a-programming-language

  2. [github_markdown] setup-habits — Programming Language Resources
     https://github.com/jwasham/coding-interview-university/blob/master/programming-language-resources.md

  3. [github_repo] setup-habits — CIU Flashcards Repo
     https://github.com/jwasham/computer-science-flash-cards

  4. [leetcode] search-bitwise — LeetCode Binary Search Blueprint
     https://leetcode.com/discuss/general-discussion/786126/python-powerful-ultimate-binary-search-template-solved-many-problems

  5. [github_pdf] search-bitwise — Bits Cheat Sheet
     https:

,url,url_type,module_id,module_title,module_phase,label
0,https://github.com/jwasham/coding-interview-university/blob/master/README.md#choose-a-programmin...,github_markdown,setup-habits,"Setup: language, cadence, and problem practice",Start Here,Choose a Programming Language
1,https://github.com/jwasham/coding-interview-university/blob/master/programming-language-resource...,github_markdown,setup-habits,"Setup: language, cadence, and problem practice",Start Here,Programming Language Resources
2,https://github.com/jwasham/coding-interview-university/blob/main/extras/cheat%20sheets/bits-chea...,github_pdf,search-bitwise,Binary search and bitwise operations,Core Track,Bits Cheat Sheet
3,https://github.com/jwasham/coding-interview-university/blob/main/extras/cheat%20sheets/system-de...,github_pdf,system-design,Optional: system design and scalability,Optional Advanced,CIU System Design Cheat Sheet
4,https://github.com/jwasham/computer-science-flash-cards,github_repo,setup-habits,"Setup: language, cadence, and problem practice",Start Here,CIU Flashcards Repo
5,https://github.com/ashishps1/awesome-behavioral-interviews,github_repo,supplemental-behavioral-interviews,Supplemental: behavioral interview repositories,Supplemental,Ashish PS1 Awesome Behavioral Interviews
6,https://github.com/sudheerj/reactjs-interview-questions,github_repo,supplemental-frontend-interviews,Supplemental: front-end interview repositories,Supplemental,Sudheer ReactJS Interview Questions
7,https://github.com/yangshun/front-end-interview-handbook,github_repo,supplemental-frontend-interviews,Supplemental: front-end interview repositories,Supplemental,Yangshun Front End Interview Handbook
8,https://github.com/Chanda-Abdul/Several-Coding-Patterns-for-Solving-Data-Structures-and-Algorith...,github_repo,supplemental-interview-handbooks,Supplemental: interview handbooks and coding patterns,Supplemental,Chanda Abdul Coding Patterns
9,https://github.com/yangshun/tech-interview-handbook,github_repo,supplemental-interview-handbooks,Supplemental: interview handbooks and coding patterns,Supplemental,Yangshun Tech Interview Handbook


## Export Markdown from all GitHub repo sources (script)

If you like the clone+glob repo handling, run the script to export markdown for **all** `github_repo` sources in the curriculum.

```bash
python3 pipeline/clone_github_repo_markdown.py --out output/github-repo-markdown
```

This writes `output/github-repo-markdown/manifest.json` plus one folder per repo.


## Process GitHub repo URLs → export all Markdown files

This runs `pipeline/clone_github_repo_markdown.py` to clone every `github_repo` source and export `*.md`/`*.mdx` into an output folder.


In [32]:
from __future__ import annotations

import json
import subprocess
from pathlib import Path

OUT_DIR = REPO_ROOT / "output" / "github-repo-markdown"
MAX_FILE_KB = 512
RECLONE = False

cmd = [
    "python3",
    str(REPO_ROOT / "pipeline" / "clone_github_repo_markdown.py"),
    "--out",
    str(OUT_DIR),
    "--max-file-kb",
    str(MAX_FILE_KB),
]
if RECLONE:
    cmd.append("--reclone")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

manifest_path = OUT_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print("manifest:", manifest_path)
print("repo_count:", manifest.get("repo_count"))
print("max_file_kb:", manifest.get("max_file_kb"))

repos = manifest.get("repos") or []
errors = [r for r in repos if r.get("error")]
print("repos_with_errors:", len(errors))
for r in errors[:20]:
    print("-", r.get("repo_url"), "=>", r.get("error"))
if len(errors) > 20:
    print(f"... {len(errors) - 20} more")

# Quick high-signal summary (top exports)
ok = [r for r in repos if not r.get("error")]
ok_sorted = sorted(ok, key=lambda r: int(r.get("exported_files") or 0), reverse=True)
print("\nTop repos by exported markdown files:")
for r in ok_sorted[:15]:
    print(
        f"- {int(r.get('exported_files') or 0):>4} files | {int(r.get('skipped_too_large') or 0):>3} skipped | {r.get('repo_url')}"
    )


Running: python3 /Users/chetangoel/Desktop/Repositories/study-app/pipeline/clone_github_repo_markdown.py --out /Users/chetangoel/Desktop/Repositories/study-app/output/github-repo-markdown --max-file-kb 512
GitHub repo sources: 13
Clone cache: /Users/chetangoel/Desktop/Repositories/study-app/pipeline/.cache/repo-markdown-clones
Output: /Users/chetangoel/Desktop/Repositories/study-app/output/github-repo-markdown
[ 1/13] https://github.com/jwasham/computer-science-flash-cards


/Users/chetangoel/Desktop/Repositories/study-app/pipeline/clone_github_repo_markdown.py:247: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": __import__("datetime").datetime.utcnow().isoformat() + "Z",
Cloning into '/Users/chetangoel/Desktop/Repositories/study-app/pipeline/.cache/repo-markdown-clones/jwasham__computer-science-flash-cards'...


[ 2/13] https://github.com/donnemartin/system-design-primer


Cloning into '/Users/chetangoel/Desktop/Repositories/study-app/pipeline/.cache/repo-markdown-clones/donnemartin__system-design-primer'...


[ 3/13] https://github.com/nas5w/interview-guide


...

[12/13] https://github.com/alirezadir/Machine-Learning-Interviews


Cloning into '/Users/chetangoel/Desktop/Repositories/study-app/pipeline/.cache/repo-markdown-clones/alirezadir__Machine-Learning-Interviews'...


[13/13] https://github.com/ashishps1/awesome-behavioral-interviews


Cloning into '/Users/chetangoel/Desktop/Repositories/study-app/pipeline/.cache/repo-markdown-clones/ashishps1__awesome-behavioral-interviews'...


Done: 13/13 repos exported. Manifest: /Users/chetangoel/Desktop/Repositories/study-app/output/github-repo-markdown/manifest.json
manifest: /Users/chetangoel/Desktop/Repositories/study-app/output/github-repo-markdown/manifest.json
repo_count: 13
max_file_kb: 512
repos_with_errors: 0

Top repos by exported markdown files:
-  287 files |   0 skipped | https://github.com/yangshun/front-end-interview-handbook
-   73 files |   0 skipped | https://github.com/yangshun/tech-interview-handbook
-   32 files |   0 skipped | https://github.com/nas5w/interview-guide
-   27 files |   0 skipped | https://github.com/alirezadir/Machine-Learning-Interviews
-   23 files |   0 skipped | https://github.com/donnemartin/system-design-primer
-   17 files |   0 skipped | https://github.com/Chanda-Abdul/Several-Coding-Patterns-for-Solving-Data-Structures-and-Algorithms-Problems-during-Interviews
-   12 files |   0 skipped | https://github.com/khangich/machine-learning-interview
-    6 files |   0 skipped | https

## Remaining URLs (non-GitHub-repo)

This lists the unique remote URLs that are **not** `github_repo` (since repos are handled via markdown export), and also applies `pipeline_url_excluded_from_stage1_seeds`.


In [34]:
from __future__ import annotations

from collections import Counter, OrderedDict
from urllib.parse import urlparse

import sys

sys.path.insert(0, str(REPO_ROOT / "pipeline"))
from config import pipeline_url_excluded_from_stage1_seeds
from url_classifier import UrlType, classify


def _dedupe_in_order(items: list[dict], key: str) -> list[dict]:
    seen: set[str] = set()
    out: list[dict] = []
    for item in items:
        k = str(item.get(key) or "").strip()
        if not k or k in seen:
            continue
        seen.add(k)
        out.append(item)
    return out


candidates: list[dict] = []
for r in rows:
    url = (r.get("url") or "").strip()
    if not url:
        continue
    if r.get("scrape_kind") != "remote_url":
        continue
    if pipeline_url_excluded_from_stage1_seeds(url):
        continue

    t = classify(url)
    if t == UrlType.GITHUB_REPO:
        continue

    candidates.append(
        {
            "url": url,
            "url_type": t.value,
            "module_id": r.get("module_id"),
            "phase": r.get("phase"),
            "label": r.get("label"),
        }
    )

remaining = _dedupe_in_order(candidates, "url")

print(f"Remaining non-github_repo URLs (deduped): {len(remaining)}\n")

by_type = Counter(item["url_type"] for item in remaining)
print("By url_type:")
for k, v in sorted(by_type.items(), key=lambda kv: (-kv[1], kv[0])):
    print(f"- {v:>4}  {k}")

hosts = Counter(urlparse(item["url"]).netloc.lower() for item in remaining)
print("\nTop hosts:")
for host, n in hosts.most_common(20):
    print(f"- {n:>4}  {host or '(empty)'}")

print("\nList:")
for i, item in enumerate(remaining, start=1):
    print(f"{i:3}. [{item['url_type']}] {item['module_id']} — {item.get('label')}")
    print(f"     {item['url']}\n")


Remaining non-github_repo URLs (deduped): 89

By url_type:
-   45  article
-   22  youtube_video
-    8  coursera
-    4  youtube_playlist
-    3  platform
-    2  github_markdown
-    2  github_pdf
-    1  archive
-    1  leetcode
-    1  shortlink

Top hosts:
-   25  www.youtube.com
-   25  en.wikipedia.org
-    8  www.coursera.org
-    4  github.com
-    3  www.geeksforgeeks.org
-    3  www.freecodecamp.org
-    2  www.topcoder.com
-    2  www.programiz.com
-    1  geni.us
-    1  startupnextdoor.com
-    1  bigocheatsheet.com
-    1  www.eskimo.com
-    1  leetcode.com
-    1  graphics.stanford.edu
-    1  youtu.be
-    1  www.amazon.com
-    1  archive.org
-    1  www.khanacademy.org
-    1  www.techinterviewhandbook.org
-    1  davidbyttow.medium.com

List:
  1. [github_markdown] setup-habits — Choose a Programming Language
     https://github.com/jwasham/coding-interview-university/blob/master/README.md#choose-a-programming-language

  2. [github_markdown] setup-habits — Program

## YouTube transcript parsing (workbench)

This section mirrors the YouTube transcript logic in `pipeline/scraper.py` so we can iterate on transcript quality (timeouts, trimming, line breaks, chunking) in a notebook.


In [35]:
from __future__ import annotations

import re
from urllib.parse import parse_qs, urlparse

from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import NoTranscriptFound, TranscriptsDisabled


In [36]:
YOUTUBE_URL = "https://www.youtube.com/watch?v=iOq5kSKqeR4"

# Optionally trim transcript to start at some time, like "1m30s" or "90".
TRIM_FROM: str | None = None

MAX_CHARS = 8000


In [37]:
def youtube_video_id(url: str) -> str:
    parsed = urlparse(url)
    if parsed.netloc.lower() == "youtu.be":
        return parsed.path.lstrip("/")
    query = parse_qs(parsed.query)
    video_ids = query.get("v")
    if not video_ids:
        raise ValueError(f"Missing YouTube video ID for URL: {url}")
    return video_ids[0]


def parse_seconds(value: str) -> int:
    value = value.strip().lower()
    if value.isdigit():
        return int(value)
    if value.endswith("s") and value[:-1].isdigit():
        return int(value[:-1])

    total = 0
    matches = re.findall(r"(\d+)([hms])", value)
    for amount, unit in matches:
        number = int(amount)
        if unit == "h":
            total += number * 3600
        elif unit == "m":
            total += number * 60
        else:
            total += number
    return total


def trim_transcript(items: list[dict], start_seconds: int) -> list[dict]:
    trimmed: list[dict] = []
    for item in items:
        start = float(item.get("start", 0))
        duration = float(item.get("duration", 0))
        if start + duration >= start_seconds:
            trimmed.append(item)
    return trimmed


In [38]:
def fetch_youtube_transcript(video_id: str) -> list[dict]:
    """Return list[{text,start,duration}] in the same shape we store in the pipeline."""
    api = YouTubeTranscriptApi()

    # Newer versions expose an instance method `.fetch()`.
    if hasattr(api, "fetch"):
        fetched = api.fetch(video_id, languages=("en",))
        out: list[dict] = []
        for item in fetched:
            if hasattr(item, "to_raw_data"):
                out.append(item.to_raw_data())
            else:
                out.append(
                    {
                        "text": str(getattr(item, "text", "")),
                        "start": float(getattr(item, "start", 0)),
                        "duration": float(getattr(item, "duration", 0)),
                    }
                )
        return out

    # Older versions expose a classmethod `get_transcript`.
    if hasattr(YouTubeTranscriptApi, "get_transcript"):
        return YouTubeTranscriptApi.get_transcript(video_id)

    raise RuntimeError("youtube-transcript-api does not expose a supported transcript API")


In [39]:
vid = youtube_video_id(YOUTUBE_URL)
print("video_id:", vid)

try:
    transcript_items = fetch_youtube_transcript(vid)
except (NoTranscriptFound, TranscriptsDisabled) as exc:
    transcript_items = []
    print("transcript_unavailable:", type(exc).__name__)

if TRIM_FROM:
    transcript_items = trim_transcript(transcript_items, parse_seconds(TRIM_FROM))

text = " ".join((item.get("text") or "").strip() for item in transcript_items).strip()
text = re.sub(r"\s+", " ", text)
text = text[:MAX_CHARS]

print("items:", len(transcript_items))
print("chars:", len(text))
print("\n--- preview ---\n")
print(text[:2000] + ("…" if len(text) > 2000 else ""))


video_id: iOq5kSKqeR4
items: 219
chars: 8000

--- preview ---

You've probably heard people talk about a fast or efficient algorithm for executing your particular task, but what exactly does it mean for an algorithm to be fast or efficient? Well, it's not talking about a measurement in real time, like seconds or minutes. This is because computer hardware and software vary drastically. My program might run slower than yours, because I'm running it on an older computer, or because I happen to be playing an online video game at the same time, which is hogging all my memory, or I might be running my program through different software which communicates with the machine differently at a low level. It's like comparing apples and oranges. Just because my slower computer takes longer than yours to give back an answer doesn't mean you have the more efficient algorithm. >> So, since we can't directly compare the runtimes of programs in seconds or minutes, how do we compare 2 different algorithms

In [40]:
# Direct-style usage (matches your snippet): chunk.text

ytt = YouTubeTranscriptApi()

try:
    chunks = ytt.fetch(vid, languages=("en",)) if hasattr(ytt, "fetch") else []
except Exception as exc:
    chunks = []
    print("fetch_failed:", type(exc).__name__, str(exc)[:200])

if chunks:
    direct_text = " ".join((getattr(chunk, "text", "") or "").strip() for chunk in chunks).strip()
    direct_text = re.sub(r"\s+", " ", direct_text)
    print("chunks:", len(chunks))
    print("chars:", len(direct_text))
    print("\n--- preview ---\n")
    print(direct_text[:2000] + ("…" if len(direct_text) > 2000 else ""))
else:
    print("No chunks (either fetch() not available or transcript unavailable).")


chunks: 219
chars: 9793

--- preview ---

You've probably heard people talk about a fast or efficient algorithm for executing your particular task, but what exactly does it mean for an algorithm to be fast or efficient? Well, it's not talking about a measurement in real time, like seconds or minutes. This is because computer hardware and software vary drastically. My program might run slower than yours, because I'm running it on an older computer, or because I happen to be playing an online video game at the same time, which is hogging all my memory, or I might be running my program through different software which communicates with the machine differently at a low level. It's like comparing apples and oranges. Just because my slower computer takes longer than yours to give back an answer doesn't mean you have the more efficient algorithm. >> So, since we can't directly compare the runtimes of programs in seconds or minutes, how do we compare 2 different algorithms regardless of their 

## Export transcripts for all YouTube video sources

This scans the curriculum `rows` for `youtube_video` URLs, fetches transcripts, and saves them under `output/youtube-transcripts/` with a `manifest.json`.

Tip: start with a small `MAX_VIDEOS` to validate quality and avoid long runs.


In [58]:
from __future__ import annotations

import json
import time
from collections import OrderedDict
from pathlib import Path

# Output folder
YOUTUBE_OUT_DIR = REPO_ROOT / "output" / "youtube-transcripts"
YOUTUBE_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Safety controls
MAX_VIDEOS = None  # set to None for all
SLEEP_SECONDS = 0.0  # small delay between requests if you see throttling

# If True, delete any existing exports in YOUTUBE_OUT_DIR before running.
CLEAN_OUTPUT_DIR = True

# Transcript text cap
# - Set to None to disable capping (save full transcript text).
# - Otherwise an int cap is applied.
try:
    import sys

    sys.path.insert(0, str(REPO_ROOT / "pipeline"))
    from config import MAX_TRANSCRIPT_CHARS as MAX_CHARS
except Exception:
    MAX_CHARS = 8000

# Example overrides:
# MAX_CHARS = None
# MAX_CHARS = 50000

MAX_CHARS = None
print("out_dir:", YOUTUBE_OUT_DIR)
print("max_chars:", MAX_CHARS)
print("clean_output_dir:", CLEAN_OUTPUT_DIR)


out_dir: /Users/chetangoel/Desktop/Repositories/study-app/output/youtube-transcripts
max_chars: None
clean_output_dir: True


In [59]:
# Build the unique YouTube video URL list (deduped, curriculum order).

import sys

sys.path.insert(0, str(REPO_ROOT / "pipeline"))
from url_classifier import UrlType, classify

unique: OrderedDict[str, dict] = OrderedDict()
for r in rows:
    url = (r.get("url") or "").strip()
    if not url or r.get("scrape_kind") != "remote_url":
        continue
    if url in unique:
        continue
    if classify(url) != UrlType.YOUTUBE_VIDEO:
        continue
    unique[url] = {
        "url": url,
        "module_id": r.get("module_id"),
        "phase": r.get("phase"),
        "label": r.get("label"),
    }

youtube_targets = list(unique.values())
if MAX_VIDEOS is not None:
    youtube_targets = youtube_targets[:MAX_VIDEOS]

print("youtube_video_urls:", len(youtube_targets))
for item in youtube_targets[:15]:
    print("-", item["module_id"], "|", item["label"], "|", item["url"])
if len(youtube_targets) > 15:
    print(f"... {len(youtube_targets) - 15} more")


youtube_video_urls: 22
- big-o | Harvard CS50 Asymptotic Notation | https://www.youtube.com/watch?v=iOq5kSKqeR4
- big-o | Big O Notations Quick Tutorial | https://www.youtube.com/watch?v=V6mKVRU1evU
- big-o | Big O Mathematical Explanation | https://www.youtube.com/watch?v=ei-A_wy5Yxw&index=2&list=PL1BaGV1cIH4UhkL8a9bJGG356covJ76qN
- arrays-linked-lists | Arrays CS50 | https://www.youtube.com/watch?v=tI_tIZFyKBw&t=3009s
- arrays-linked-lists | Linked Lists CS50 | https://www.youtube.com/watch?v=2T-A_GFuoTo&t=650s
- stacks-queues-hashes | Hashing with Chaining | https://www.youtube.com/watch?v=0M_kIqhwbFo&list=PLUl4u3cNGP61Oq3tWYp6V_F-5jb5L2iHb&index=8
- stacks-queues-hashes | The Mighty Dictionary | https://www.youtube.com/watch?v=C4Kc8xzcA68
- search-bitwise | Bit Manipulation Intro | https://www.youtube.com/watch?v=7jkIUgLC29I
- trees-heaps | Binary Search Tree Review | https://www.youtube.com/watch?v=x6At0nzX92o&index=1&list=PLA5Lqm4uh9Bbq-E0ZnqTIa8LRaL77ica6
- trees-heaps | MIT Bin

In [60]:
# Fetch and save transcripts.

import re
from urllib.parse import parse_qs, urlparse

from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import NoTranscriptFound, TranscriptsDisabled


def youtube_video_id(url: str) -> str:
    parsed = urlparse(url)
    if parsed.netloc.lower() == "youtu.be":
        return parsed.path.lstrip("/")
    query = parse_qs(parsed.query)
    video_ids = query.get("v")
    if not video_ids:
        raise ValueError(f"Missing YouTube video ID for URL: {url}")
    return video_ids[0]


def fetch_items(video_id: str) -> list[dict]:
    api = YouTubeTranscriptApi()
    if hasattr(api, "fetch"):
        fetched = api.fetch(video_id, languages=("en",))
        out: list[dict] = []
        for item in fetched:
            if hasattr(item, "to_raw_data"):
                out.append(item.to_raw_data())
            else:
                out.append(
                    {
                        "text": str(getattr(item, "text", "")),
                        "start": float(getattr(item, "start", 0)),
                        "duration": float(getattr(item, "duration", 0)),
                    }
                )
        return out
    if hasattr(YouTubeTranscriptApi, "get_transcript"):
        return YouTubeTranscriptApi.get_transcript(video_id)
    raise RuntimeError("youtube-transcript-api does not expose a supported transcript API")


def items_to_text(items: list[dict], *, max_chars: int | None) -> str:
    text = " ".join((it.get("text") or "").strip() for it in items).strip()
    text = re.sub(r"\s+", " ", text)
    if max_chars is None:
        return text
    if int(max_chars) <= 0:
        return text
    return text[: int(max_chars)]


manifest = {
    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "count": len(youtube_targets),
    "max_chars": MAX_CHARS,
    "items": [],
}

# Optionally clear old exports so re-runs are clean.
if CLEAN_OUTPUT_DIR:
    removed = 0
    for p in YOUTUBE_OUT_DIR.glob("*.json"):
        p.unlink(missing_ok=True)
        removed += 1
    for p in YOUTUBE_OUT_DIR.glob("*.txt"):
        p.unlink(missing_ok=True)
        removed += 1
    (YOUTUBE_OUT_DIR / "manifest.json").unlink(missing_ok=True)
    print(f"cleaned_files: {removed}")

ok = 0
for i, t in enumerate(youtube_targets, start=1):
    url = t["url"]
    vid = None
    try:
        vid = youtube_video_id(url)
        items = fetch_items(vid)
        text = items_to_text(items, max_chars=MAX_CHARS)

        out_json = YOUTUBE_OUT_DIR / f"{vid}.json"
        out_txt = YOUTUBE_OUT_DIR / f"{vid}.txt"
        out_json.write_text(json.dumps({"url": url, "video_id": vid, "items": items}, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        out_txt.write_text(text + "\n", encoding="utf-8")

        ok += 1
        manifest["items"].append(
            {
                **t,
                "video_id": vid,
                "status": "ok",
                "items": len(items),
                "chars": len(text),
                "json": out_json.name,
                "text": out_txt.name,
            }
        )
        print(f"[{i:>2}/{len(youtube_targets)}] OK  {vid}  items={len(items)} chars={len(text)}")
    except (NoTranscriptFound, TranscriptsDisabled) as exc:
        manifest["items"].append({**t, "video_id": vid, "status": "no_transcript", "error": type(exc).__name__})
        print(f"[{i:>2}/{len(youtube_targets)}] NO_TRANSCRIPT {vid or '?'}  {type(exc).__name__}")
    except Exception as exc:
        manifest["items"].append({**t, "video_id": vid, "status": "failed", "error": f"{type(exc).__name__}: {exc}"})
        print(f"[{i:>2}/{len(youtube_targets)}] FAIL {vid or '?'}  {type(exc).__name__}")

    if SLEEP_SECONDS:
        time.sleep(SLEEP_SECONDS)

manifest_path = YOUTUBE_OUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print("\nSummary")
print("- ok:", ok)
print("- total:", len(youtube_targets))
print("- manifest:", manifest_path)


cleaned_files: 6
[ 1/22] OK  iOq5kSKqeR4  items=219 chars=9793
[ 2/22] OK  V6mKVRU1evU  items=540 chars=20859
[ 3/22] OK  ei-A_wy5Yxw  items=932 chars=27919
[ 4/22] OK  tI_tIZFyKBw  items=4544 chars=142717
[ 5/22] OK  2T-A_GFuoTo  items=4762 chars=150849
[ 6/22] OK  0M_kIqhwbFo  items=949 chars=37032
[ 7/22] OK  C4Kc8xzcA68  items=752 chars=28115
[ 8/22] OK  7jkIUgLC29I  items=712 chars=23566
[ 9/22] OK  x6At0nzX92o  items=255 chars=9280
[10/22] OK  Xnpo1atN-Iw  items=953 chars=37803
[11/22] NO_TRANSCRIPT kPRA0W1kECg  NoTranscriptFound
[12/22] OK  oFVYVzlvk9c  items=1209 chars=49546
[13/22] OK  IBfWDYSffUU  items=1232 chars=45854
[14/22] OK  Sjk0xqWWPCc  items=1880 chars=53923
[15/22] OK  gl3emqCuueQ  items=1200 chars=44394
[16/22] OK  ngCos392W4w  items=494 chars=18492
[17/22] OK  wAA0AMfcJHQ  items=1949 chars=55526
[18/22] OK  XM4lGflQFvA  items=105 chars=3907
[19/22] OK  vjYF_fAZI5E  items=1287 chars=48426
[20/22] OK  SAhJf36_u5U  items=1348 chars=48365
[21/22] OK  r4r1DZcx1cM  item

## Crawl remaining URLs with Crawl4AI

This runs `pipeline/crawl_remaining_with_crawl4ai.py` and saves markdown + a manifest under `output/crawl4ai-remaining/`.

```bash
python3 pipeline/crawl_remaining_with_crawl4ai.py --clean --max-urls 15
# then full run:
python3 pipeline/crawl_remaining_with_crawl4ai.py --clean
```


## Run Crawl4AI remaining crawl (from notebook)

This runs the batch crawler and then prints a quick summary from `manifest.json`.


In [62]:
from __future__ import annotations

import json
import subprocess
from pathlib import Path

CRAWL4AI_OUT_DIR = REPO_ROOT / "output" / "crawl4ai-remaining"
MAX_URLS = 15  # set to 0 for all
CLEAN = True
SKIP_EXISTING = True  # if True, won't re-crawl URLs already in output

cmd = [
    "python3",
    str(REPO_ROOT / "pipeline" / "crawl_remaining_with_crawl4ai.py"),
    "--out",
    str(CRAWL4AI_OUT_DIR),
]
if CLEAN:
    cmd.append("--clean")
if SKIP_EXISTING and not CLEAN:
    cmd.append("--skip-existing")
if MAX_URLS and MAX_URLS > 0:
    cmd += ["--max-urls", str(MAX_URLS)]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

manifest_path = CRAWL4AI_OUT_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
items = manifest.get("items") or []

ok = [it for it in items if it.get("status") == "ok"]
empty = [it for it in items if it.get("status") == "empty"]
failed = [it for it in items if it.get("status") == "failed"]

print("\nSummary")
print("- count:", manifest.get("count"))
print("- ok:", len(ok))
print("- empty:", len(empty))
print("- failed:", len(failed))
print("- manifest:", manifest_path)

if failed:
    print("\nFirst failures:")
    for it in failed[:15]:
        print("-", it.get("url_type"), it.get("url"), "=>", (it.get("error") or "")[:160])


Running: python3 /Users/chetangoel/Desktop/Repositories/study-app/pipeline/crawl_remaining_with_crawl4ai.py --out /Users/chetangoel/Desktop/Repositories/study-app/output/crawl4ai-remaining --clean --max-urls 15
Remaining URLs to crawl: 15
By type: {'article': 4, 'coursera': 6, 'github_markdown': 2, 'github_pdf': 1, 'leetcode': 1, 'youtube_playlist': 1}
Output: /Users/chetangoel/Desktop/Repositories/study-app/output/crawl4ai-remaining
[  1/15] github_markdown https://github.com/jwasham/coding-interview-university/blob/master/README.md#choose-a-programming-language
[INIT].... → Crawl4AI 0.8.6 
[FETCH]... ↓ 
https://github.com/jwasham/coding-interview-univ...b/master/README.md#choose-a-p
rogramming-language  | ✓ | ⏱: 2.33s 
[SCRAPE].. ◆ 
https://github.com/jwasham/coding-interview-univ...b/master/README.md#choose-a-p
rogramming-language  | ✓ | ⏱: 0.22s 
[COMPLETE] ● 
https://github.com/jwasham/coding-interview-univ...b/master/README.md#choose-a-p
rogramming-language  | ✓ | ⏱: 2.57s 
[  2/